In [1]:
import os

# 查看当前工作目录
print(os.getcwd())

# 更改工作目录（Windows 示例）
os.chdir(r"D:\excel_clean")

# Linux/macOS 示例
# os.chdir("/home/username/documents")

# 确认更改后的目录
print(os.getcwd())

D:\Jupyter notebook 代码
D:\excel_clean


In [7]:
# Cell 1: 导入与全局配置
import re
import logging
from pathlib import Path
import numpy as np
import pandas as pd

# 路径配置
INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")
OUTPUT_FILE = OUTPUT_DIR / "汇总清洗后.xlsx"
LOG_FILE = OUTPUT_DIR / "清洗日志.log"

# 辅助列配置
AUX_COLS = ["_来源文件", "_来源sheet", "_来源行号"]

# 定义各表的必须列名（用于自动识别 sheet 角色）
ORDERS_COLS = {"Order_id", "date", "customer_id", "product_id", "quantity"}
PRODUCTS_COLS = {"product_id", "product_name"}
CUSTOMERS_COLS = {"customer_id", "customer_name"}

def setup_logging():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(LOG_FILE, encoding="utf-8"),
            logging.StreamHandler(),
        ],
        force=True, # 适配 Jupyter 重复运行
    )
    print("配置加载完成")

setup_logging()

配置加载完成


In [8]:
# Cell 2: 清洗工具函数
def clean_quantity(val):
    """提取数量中的数字，返回 (清洗后的数值, 是否异常标记)"""
    if pd.isna(val):
        return None, False
    s = str(val).strip()
    if s == "":
        return None, False
    
    # 匹配数字部分，比如 13Q -> 13
    match = re.search(r"\d+", s)
    if match:
        num = int(match.group())
        # 如果原字符串包含非数字字符，则标记为 True
        is_abnormal = not s.isdigit()
        return num, is_abnormal
    return None, True # 如果连数字都没有，置空且标记异常

def clean_price(val):
    """清洗价格，去除非数字和小数点字符，转为 float"""
    if pd.isna(val):
        return np.nan
    s = str(val)
    # 移除 ₹, 逗号 等非数字/小数点字符
    s = re.sub(r"[^\d.]", "", s)
    try:
        return float(s)
    except ValueError:
        return np.nan

def clean_product_name(val):
    """清理商品名前缀，如 '1. Apples' -> 'Apples'"""
    if pd.isna(val):
        return val
    s = str(val).strip()
    # 正则匹配 "数字. 空格" 开头并去除
    s = re.sub(r"^\d+\.\s*", "", s)
    return s

def is_date_column(col):
    c = str(col).strip()
    if "日期" in c: return True
    c2 = re.sub(r"(?<!^)(?=[A-Z])", "_", c).lower()
    return any(t in {"date", "datetime"} for t in re.split(r"[_\s\-]+", c2))

print("工具函数定义完成")

工具函数定义完成


In [9]:
# Cell 3: 读取文件并分类
def read_and_classify_files():
    files = sorted(
        [p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".xlsx" and not p.name.startswith("~$")],
        key=lambda p: p.name
    )
    if not files:
        logging.error("input 目录下没有找到 .xlsx 文件")
        return None, None, None

    orders_dfs, products_dfs, customers_dfs = [], [], []

    for f in files:
        try:
            sheets = pd.read_excel(f, sheet_name=None, engine="openpyxl", dtype=object)
        except Exception as e:
            logging.error(f"读取文件失败 {f.name}: {e}")
            continue

        for sheet_name, df in sheets.items():
            if df.empty: continue
            
            df = df.copy()
            df.columns = [str(c).strip() for c in df.columns]
            
            # 添加辅助列
            df["_来源文件"] = f.name
            df["_来源sheet"] = sheet_name
            df["_来源行号"] = df.index + 2
            
            cols_set = set(df.columns)
            
            # 基于列名特征识别表角色
            if ORDERS_COLS.issubset(cols_set):
                orders_dfs.append(df)
                logging.info(f"识别为 orders: {f.name} / {sheet_name} ({len(df)}行)")
            elif PRODUCTS_COLS.issubset(cols_set):
                products_dfs.append(df)
                logging.info(f"识别为 products: {f.name} / {sheet_name} ({len(df)}行)")
            elif CUSTOMERS_COLS.issubset(cols_set):
                customers_dfs.append(df)
                logging.info(f"识别为 customers: {f.name} / {sheet_name} ({len(df)}行)")
            else:
                logging.warning(f"无法识别表结构，已跳过: {f.name} / {sheet_name}")

    # 拼接同类表
    df_orders = pd.concat(orders_dfs, ignore_index=True) if orders_dfs else pd.DataFrame()
    df_products = pd.concat(products_dfs, ignore_index=True) if products_dfs else pd.DataFrame()
    df_customers = pd.concat(customers_dfs, ignore_index=True) if customers_dfs else pd.DataFrame()
    
    return df_orders, df_products, df_customers

df_orders, df_products, df_customers = read_and_classify_files()
print(f"合并前: Orders {len(df_orders)} 行, Products {len(df_products)} 行, Customers {len(df_customers)} 行")

2026-11-15 22:39:18,477 [INFO] 识别为 orders: UnClean_Product_sales_RowData.xlsx / orders (310行)
2026-11-15 22:39:18,478 [INFO] 识别为 products: UnClean_Product_sales_RowData.xlsx / products (50行)
2026-11-15 22:39:18,480 [INFO] 识别为 customers: UnClean_Product_sales_RowData.xlsx / customers (90行)


合并前: Orders 310 行, Products 50 行, Customers 90 行


In [10]:
# Cell 4: 各表清洗与去重
if df_orders.empty:
    raise ValueError("没有读取到任何 orders 数据，终止执行")

# ============ 1. 清洗 orders ============
# 删全空行（排除辅助列）
data_cols = [c for c in df_orders.columns if c not in AUX_COLS]
df_orders = df_orders[~df_orders[data_cols].replace(r"^\s*$", np.nan, regex=True).isna().all(axis=1)].copy()

# 清理 quantity，并生成异常标记
q_results = df_orders["quantity"].apply(clean_quantity)
df_orders["quantity"] = [r[0] for r in q_results]
df_orders["_quantity_异常标记"] = [r[1] for r in q_results]

# 清理日期
for col in df_orders.columns:
    if is_date_column(col):
        dt = pd.to_datetime(df_orders[col].astype(str).replace({"nan": None, "None": None}), errors="coerce")
        df_orders[col] = dt.dt.strftime("%Y-%m-%d")

# 统一 ID 类型（转为 pandas 数值型，支持 null）
for col in ["Order_id", "customer_id", "product_id"]:
    if col in df_orders.columns:
        df_orders[col] = pd.to_numeric(df_orders[col], errors="coerce")

# Orders 去重 (Order_id, product_id)
before = len(df_orders)
# 保留 order_id 和 product_id 都不为空的基础行进行去重
df_orders = df_orders.drop_duplicates(subset=["Order_id", "product_id"], keep="first").copy()
logging.info(f"Orders 去重: 删除了 {before - len(df_orders)} 行")


# ============ 2. 清洗 products ============
if not df_products.empty:
    data_cols = [c for c in df_products.columns if c not in AUX_COLS]
    df_products = df_products[~df_products[data_cols].replace(r"^\s*$", np.nan, regex=True).isna().all(axis=1)].copy()
    
    df_products["Price(Rs)"] = df_products["Price(Rs)"].apply(clean_price)
    df_products["product_name"] = df_products["product_name"].apply(clean_product_name)
    df_products["product_id"] = pd.to_numeric(df_products["product_id"], errors="coerce")
    
    # 维度表按主键去重
    df_products = df_products.drop_duplicates(subset=["product_id"], keep="first")


# ============ 3. 清洗 customers ============
if not df_customers.empty:
    data_cols = [c for c in df_customers.columns if c not in AUX_COLS]
    df_customers = df_customers[~df_customers[data_cols].replace(r"^\s*$", np.nan, regex=True).isna().all(axis=1)].copy()
    
    df_customers["customer_id"] = pd.to_numeric(df_customers["customer_id"], errors="coerce")
    
    # 维度表按主键去重
    df_customers = df_customers.drop_duplicates(subset=["customer_id"], keep="first")

print("清洗完成。准备 Join。")

2026-11-15 22:39:33,288 [INFO] Orders 去重: 删除了 0 行


清洗完成。准备 Join。


In [11]:
# Cell 5: 三表 Join
# orders left join products
df_wide = pd.merge(
    df_orders, 
    df_products[["product_id", "product_name", "Price(Rs)"]], 
    on="product_id", 
    how="left"
)

# left join customers
df_wide = pd.merge(
    df_wide, 
    df_customers[["customer_id", "customer_name"]], 
    on="customer_id", 
    how="left"
)

# 重排列名，确保辅助列在最后，且只保留 orders 的辅助列
core_cols = [
    "Order_id", "date", "customer_id", "customer_name", 
    "product_id", "product_name", "Price(Rs)", "quantity"
]
# 只保留实际存在的 core_cols
exist_core_cols = [c for c in core_cols if c in df_wide.columns]

# 异常标记列如果有的话
extra_cols = ["_quantity_异常标记"] if "_quantity_异常标记" in df_wide.columns else []

df_wide = df_wide[exist_core_cols + extra_cols + AUX_COLS]

print(f"宽表合并完成，总行数: {len(df_wide)}")
display(df_wide.head())

宽表合并完成，总行数: 310


C:\Users\33353\AppData\Local\Temp\ipykernel_21524\2241879310.py:3: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  df_wide = pd.merge(


,Order_id,date,customer_id,customer_name,product_id,product_name,Price(Rs),quantity,_quantity_异常标记,_来源文件,_来源sheet,_来源行号
0,1236,2024-01-01,74,Arjun,148.0,Apples,150.0,13.0,False,UnClean_Product_sales_RowData.xlsx,orders,2
1,1428,2024-01-01,54,Priya,149.0,Bananas,40.0,1.0,False,UnClean_Product_sales_RowData.xlsx,orders,3
2,1585,2024-01-01,71,Liam,248.0,Oranges,80.0,3.0,False,UnClean_Product_sales_RowData.xlsx,orders,4
3,1240,2024-01-02,56,Emma,242.0,Grapes,200.0,15.0,False,UnClean_Product_sales_RowData.xlsx,orders,5
4,1311,2024-01-03,79,Ayaan,299.0,Strawberries,300.0,18.0,False,UnClean_Product_sales_RowData.xlsx,orders,6


In [12]:
# Cell 6: 导出 Excel
try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_wide.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
    print(f"✅ 输出成功: {OUTPUT_FILE.resolve()}")
except Exception as e:
    logging.exception(f"❌ 输出失败，请检查文件是否被占用: {e}")

✅ 输出成功: D:\excel_clean\output\汇总清洗后.xlsx
